# 086 — GAN y entrenamiento adversarial

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**GAN** (Goodfellow et al., 2014): dos redes en un juego de suma cero —
un generador G que transforma ruido z en muestras y un discriminador D que
estima la probabilidad de que una muestra sea real:

```text
min_G max_D  E_{x~p_data}[log D(x)] + E_{z}[log(1 − D(G(z)))]
```

**Discriminador óptimo** para G fijo: `D*(x) = p_data(x) / (p_data(x) + p_g(x))`.
Sustituyéndolo, entrenar G minimiza la **divergencia de Jensen-Shannon** entre
p_data y p_g; en el equilibrio p_g = p_data y D*(·) = 1/2 en todas partes.

**Non-saturating loss**: G maximiza log D(G(z)) en vez de minimizar
log(1 − D(G(z))) — mismo punto fijo, pero con gradiente fuerte justo cuando D
domina. Fallos característicos: **colapso de modos** (G cubre pocas modas) y
oscilación, porque el equilibrio es un punto de silla, no un mínimo.

## 🧮 Ejemplo de referencia

Minibatch con D(x) = 0.9, 0.7 en reales y D(G(z)) = 0.4, 0.2 en falsas:

- L_D = −½[log 0.9 + log 0.7] − ½[log 0.6 + log 0.8] = 0.2310 + 0.3670 = **0.5980**
- L_G non-saturating = −½[log 0.4 + log 0.2] = **1.2629** (la muestra con D = 0.2
  aporta 1.6094: más gradiente donde G más falla)
- L_G saturante = ½[log 0.6 + log 0.8] = **−0.3670**
- Si en una región p_data = 0.3 y p_g = 0.1, D*(x) = 0.3/0.4 = **0.75**.

Verifícalo a mano antes de ejecutar el laboratorio.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=86)
show(result)


## Reflexión

1. Si el discriminador alcanza rápidamente D(G(z)) ≈ 0 para todo z, ¿por qué la pérdida
   original de G deja de aprender y cómo lo corrige la variante non-saturating?
2. ¿Por qué una pérdida de entrenamiento baja del generador NO demuestra ausencia de
   colapso de modos, y qué evidencia adicional pedirías (diversidad, cobertura)?
3. Comparado con el VAE de la clase 085, la GAN no ofrece verosimilitud: ¿qué pierdes
   y qué ganas con ese intercambio?
